# Notebook 01 — Data Acquisition

**Purpose:** Download all raw data, verify integrity, document provenance.

**Outputs:**
- `data/raw/` — all downloaded files
- `data/raw/MANIFEST.json` — provenance log (URLs, checksums, descriptions)

**Data sources:**
1. USGS SAR-HiSAR Global Mosaic (351 m/pixel, ~1 GB GeoTIFF)
2. NLDSAR Denoised Dataset (Zenodo)
3. Lopes et al. (2020) Geomorphological Map (label ground truth)
4. BIDR swaths (subset covering Dragonfly landing area)

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src') if os.path.basename(os.getcwd()) == 'notebooks' else 'src')
sys.path.insert(0, os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else '.')

import json
import hashlib
import datetime
from pathlib import Path

import requests
from tqdm.notebook import tqdm

from src.utils import RAW_DIR, sha256_file, save_manifest, load_manifest, get_logger

log = get_logger('01_data_acquisition')
RAW_DIR.mkdir(parents=True, exist_ok=True)
print(f'Raw data directory: {RAW_DIR}')

Raw data directory: /home/gabriel/titan-sar/data/raw


## Helper: resumable download

In [2]:
def download_file(url, dest_path, description="", chunk_size=1 << 20):
    """Download a file with progress bar and resume support."""
    dest_path = Path(dest_path)
    if dest_path.exists():
        log.info(f'Already exists: {dest_path.name} ({dest_path.stat().st_size:,} bytes)')
        return dest_path

    log.info(f'Downloading: {description or url}')
    headers = {}
    partial = dest_path.with_suffix(dest_path.suffix + '.partial')
    if partial.exists():
        resume_pos = partial.stat().st_size
        headers['Range'] = f'bytes={resume_pos}-'
        log.info(f'Resuming from {resume_pos:,} bytes')
    else:
        resume_pos = 0

    resp = requests.get(url, headers=headers, stream=True, timeout=60)
    resp.raise_for_status()

    total = int(resp.headers.get('content-length', 0)) + resume_pos
    mode = 'ab' if resume_pos else 'wb'

    with open(partial, mode) as f, tqdm(
        total=total, initial=resume_pos, unit='B', unit_scale=True,
        desc=dest_path.name
    ) as pbar:
        for chunk in resp.iter_content(chunk_size=chunk_size):
            f.write(chunk)
            pbar.update(len(chunk))

    partial.rename(dest_path)
    log.info(f'Saved: {dest_path.name} ({dest_path.stat().st_size:,} bytes)')
    return dest_path

## 1. USGS SAR-HiSAR Global Mosaic

Single GeoTIFF, ~1 GB, 351 m/pixel, Ku-band backscatter (σ₀), grayscale.

In [3]:
# Correct URL found from USGS Astrogeology product page
SAR_MOSAIC_URL = (
    "https://planetarymaps.usgs.gov/mosaic/"
    "Titan_SAR_HiSAR_MosaicThru_T104_Jan2015_clon180_128ppd.tif"
)
SAR_MOSAIC_FILENAME = "Titan_SAR_HiSAR_Global_Mosaic_351m.tif"

sar_mosaic_path = RAW_DIR / SAR_MOSAIC_FILENAME

download_file(
    SAR_MOSAIC_URL,
    sar_mosaic_path,
    description="USGS SAR-HiSAR Global Mosaic (351 m/pixel, 1013 MB)"
)

08:41:48 [01_data_acquisition] INFO: Already exists: Titan_SAR_HiSAR_Global_Mosaic_351m.tif (1,061,868,159 bytes)


PosixPath('/home/gabriel/titan-sar/data/raw/Titan_SAR_HiSAR_Global_Mosaic_351m.tif')

In [4]:
# Verify the SAR mosaic opens correctly
import rasterio

if sar_mosaic_path.exists():
    with rasterio.open(sar_mosaic_path) as src:
        print(f'CRS:        {src.crs}')
        print(f'Resolution: {src.res}')
        print(f'Shape:      {src.height} x {src.width}')
        print(f'Bounds:     {src.bounds}')
        print(f'Dtype:      {src.dtypes}')
        print(f'NoData:     {src.nodata}')
else:
    print('SAR mosaic not yet downloaded — see above.')

CRS:        PROJCS["SimpleCylindrical Titan",GEOGCS["GCS_Titan",DATUM["D_Titan",SPHEROID["Titan",2575000,0]],PRIMEM["Reference_Meridian",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Equirectangular"],PARAMETER["standard_parallel_1",0],PARAMETER["central_meridian",180],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Resolution: (351.11115812, 351.11115812)
Shape:      23040 x 46080
Bounds:     BoundingBox(left=-8089601.082994, bottom=-4044800.5415878003, right=8089601.0831756005, top=4044800.541497)
Dtype:      ('uint8',)
NoData:     0.0


## 2. NLDSAR Denoised Dataset

Non-local means denoised SAR. Available on Zenodo: https://zenodo.org/records/528545

In [5]:
# The Zenodo record may contain multiple files. We'll fetch the record metadata
# first to discover the actual download links.

NLDSAR_ZENODO_RECORD = "528545"
NLDSAR_API_URL = f"https://zenodo.org/api/records/{NLDSAR_ZENODO_RECORD}"

nldsar_dir = RAW_DIR / "nldsar"
nldsar_dir.mkdir(exist_ok=True)

try:
    resp = requests.get(NLDSAR_API_URL, timeout=30)
    resp.raise_for_status()
    record = resp.json()
    
    print(f"Title: {record['metadata']['title']}")
    print(f"DOI:   {record['doi']}")
    print(f"Files: {len(record['files'])}")
    print()
    
    nldsar_files = []
    for f in record['files']:
        size_mb = f['size'] / 1e6
        print(f"  {f['key']:40s}  {size_mb:8.1f} MB")
        nldsar_files.append({
            'filename': f['key'],
            'url': f['links']['self'],
            'size': f['size'],
            'checksum': f['checksum'],  # md5:...
        })
except Exception as e:
    log.warning(f"Could not fetch Zenodo record: {e}")
    nldsar_files = []

Title: NLDSAR - Cassini Non-Local Denoised SAR Dataset
DOI:   10.5281/zenodo.528545
Files: 29

  t13_reproc1_corramb_bidr_nldsar_v1.cub       440.5 MB
  t23_reproc1_corramb_bidr_nldsar_v1.cub       589.9 MB
  MOSAIC_NLDSAR_88S88N_180W180E_EQUI_D128_v1.cub     201.1 MB
  t36_reproc1_corramb_bidr_nldsar_v1.cub        65.6 MB
  t39_reproc1_corramb_bidr_nldsar_v1.cub       511.3 MB
  t41_reproc1_corramb_bidr_nldsar_v1.cub       356.6 MB
  t49_reproc1_corramb_bidr_nldsar_v1.cub       251.7 MB
  ta_reproc1_corramb_bidr_nldsar_v1.cub        432.1 MB
  t8_reproc1_corramb_bidr_nldsar_v1.cub        583.1 MB
  t7_reproc1_corramb_bidr_nldsar_v1.cub        150.0 MB
  t65_reproc1_corramb_bidr_nldsar_v1.cub       597.8 MB
  t61_reproc1_corramb_bidr_nldsar_v1.cub       629.2 MB
  t59_reproc1_corramb_bidr_nldsar_v1.cub       276.9 MB
  t58_reproc1_corramb_bidr_nldsar_v1.cub       620.8 MB
  t57_reproc1_corramb_bidr_nldsar_v1.cub       462.5 MB
  t56_reproc1_corramb_bidr_nldsar_v1.cub       597.8 MB
  t

In [6]:
# Download NLDSAR files (these can be large — download selectively if needed)
for finfo in nldsar_files:
    dest = nldsar_dir / finfo['filename']
    try:
        download_file(finfo['url'], dest, description=f"NLDSAR: {finfo['filename']}")
    except Exception as e:
        log.warning(f"Failed to download {finfo['filename']}: {e}")

08:41:49 [01_data_acquisition] INFO: Already exists: t13_reproc1_corramb_bidr_nldsar_v1.cub (440,472,447 bytes)


08:41:49 [01_data_acquisition] INFO: Already exists: t23_reproc1_corramb_bidr_nldsar_v1.cub (589,894,524 bytes)


08:41:49 [01_data_acquisition] INFO: Already exists: MOSAIC_NLDSAR_88S88N_180W180E_EQUI_D128_v1.cub (201,129,984 bytes)


08:41:49 [01_data_acquisition] INFO: Already exists: t36_reproc1_corramb_bidr_nldsar_v1.cub (65,606,522 bytes)


08:41:49 [01_data_acquisition] INFO: Already exists: t39_reproc1_corramb_bidr_nldsar_v1.cub (511,251,328 bytes)


...

08:41:49 [01_data_acquisition] INFO: Already exists: t28_reproc1_corramb_bidr_nldsar_v1.cub (613,487,482 bytes)


08:41:49 [01_data_acquisition] INFO: Already exists: t18_reproc1_corramb_bidr_nldsar_v1.cub (199,299,961 bytes)


08:41:49 [01_data_acquisition] INFO: Already exists: t17_reproc1_corramb_bidr_nldsar_v1.cub (92,345,206 bytes)


08:41:49 [01_data_acquisition] INFO: Already exists: t29_reproc1_corramb_bidr_nldsar_v1.cub (578,098,043 bytes)


08:41:49 [01_data_acquisition] INFO: Already exists: t16_reproc1_corramb_bidr_nldsar_v1.cub (597,758,844 bytes)


## 3. Lopes et al. (2020) Geomorphological Map

This is the **single most important file** — it provides ground-truth training labels.

Six classes: plains, dunes, hummocky/mountainous, lakes/seas, labyrinth, craters.

**Search strategy:**
1. Check USGS Astrogeology for Titan geomorphological map products
2. Check the Nature Astronomy supplementary materials (doi:10.1038/s41550-019-0917-6)
3. Check public data repositories (Zenodo, Figshare, etc.)
4. If unavailable as raster, plan to digitise from published figure

In [7]:
# The Lopes et al. (2020) geomorphological map GIS data is hosted at
# ASU's Ronald Greeley Center for Planetary Studies
# Reference: Lopes et al., 2020, Nature Astronomy, 4, 228-233

import requests as _req

geomorph_dir = RAW_DIR / 'geomorphology'
geomorph_dir.mkdir(exist_ok=True)

TITAN_GEOMAP_URL = "https://rgcps.asu.edu/gisdata/TITAN_2019-11_global_geomap_6unit.zip"
geomap_zip = geomorph_dir / "TITAN_2019-11_global_geomap_6unit.zip"

download_file(TITAN_GEOMAP_URL, geomap_zip,
              description="Titan 6-unit Geomorphological Map (ASU/RGCPS, 2.36 GB)")

08:41:49 [01_data_acquisition] INFO: Already exists: TITAN_2019-11_global_geomap_6unit.zip (2,363,135,078 bytes)


PosixPath('/home/gabriel/titan-sar/data/raw/geomorphology/TITAN_2019-11_global_geomap_6unit.zip')

In [8]:
# Extract the geomorphological map zip
import zipfile

if geomap_zip.exists():
    extract_dir = geomorph_dir / 'titan_6unit_geomap'
    if not extract_dir.exists():
        with zipfile.ZipFile(geomap_zip) as zf:
            print(f"Contents of {geomap_zip.name}:")
            for info in sorted(zf.infolist(), key=lambda x: x.filename)[:20]:
                print(f"  {info.filename:60s}  {info.file_size:>12,} bytes")
            zf.extractall(extract_dir)
            print(f"\nExtracted to: {extract_dir}")
    else:
        print(f"Already extracted: {extract_dir}")
else:
    print("Geomorphological map zip not available.")

Already extracted: /home/gabriel/titan-sar/data/raw/geomorphology/titan_6unit_geomap


In [9]:
# Inspect the geodatabase (ESRI FileGDB format)
import fiona

gdb_path = geomorph_dir / 'titan_6unit_geomap' / 'TITAN_2019-11_global_geomap_6unit' / 'Titan_Geodatabase_2019-11.gdb'

if gdb_path.exists():
    layers = fiona.listlayers(str(gdb_path))
    print(f"Geodatabase layers: {layers}\n")
    
    # Focus on the geologic units layer
    with fiona.open(str(gdb_path), layer='TITAN_GeologicUnits') as src:
        print(f"Layer: TITAN_GeologicUnits")
        print(f"  CRS:      {src.crs}")
        print(f"  Features: {len(src)}")
        print(f"  Schema:   {dict(src.schema['properties'])}")
        print(f"\n  Terrain classes:")
        for feat in src:
            name = feat['properties']['Meta_Terra']
            area = feat['properties']['Shape_Area']
            print(f"    {name:20s}  area={area:.2e} m²")
else:
    print(f"Geodatabase not found at {gdb_path}")

Geodatabase layers: ['TITAN_nomenclature', 'TITAN_nomenclature_13names', 'TITAN_graticule_30d_clon180', 'TITAN_graticule_10d_clon180', 'TITAN_Boundary_SPole_stereographic', 'TITAN_Boundary_NPole_stereographic', 'TITAN_Boundary_Merc', 'TITAN_GeologicUnits']

Layer: TITAN_GeologicUnits


  CRS:      PROJCS["TITAN_PlateCarree_c180",GEOGCS["GCS_TITAN",DATUM["D_TITAN",SPHEROID["TITAN_localRadius",2575000,0]],PRIMEM["Reference_Meridian",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Equirectangular"],PARAMETER["central_meridian",180],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
  Features: 9
  Schema:   {'Meta_Terra': 'str:50', 'Shape_Leng': 'float', 'Shape_Length': 'float', 'Shape_Area': 'float'}

  Terrain classes:
    Basins                area=4.70e+12 m²
    Craters               area=3.73e+11 m²
    Dunes                 area=1.51e+13 m²
    Labyrinth             area=5.11e+12 m²


    Mountains             area=1.58e+13 m²
    Plains                area=8.96e+13 m²
    Plains                area=1.18e+11 m²
    Plains                area=8.88e+10 m²
    Plains                area=9.83e+10 m²


## 4. BIDR Swaths (Dragonfly Landing Region)

Individual SAR swaths at ~175 m/pixel. Targeting the Shangri-La / Selk Crater region
(approx. 0°–10°N, 160°–200°W).

BIDR products are on PDS: https://pds-imaging.jpl.nasa.gov/volumes/radar.html

In [10]:
# Key Cassini flybys covering the Dragonfly landing region:
# T8 (CORADR_0048), T28 (CORADR_0101), T29 (CORADR_0102), 
# T41 (CORADR_0138), T83 (CORADR_0211), T95 (CORADR_0225)
# These have good SAR coverage of Shangri-La and Selk Crater.

PDS_BASE_URL = "https://pds-imaging.jpl.nasa.gov/data/cassini/cassini_orbiter/"

# We'll target a small subset to keep download manageable
BIDR_TARGETS = [
    {
        'volume': 'CORADR_0048',
        'flyby': 'T8',
        'description': 'T8 flyby — Shangri-La dune field coverage',
    },
    {
        'volume': 'CORADR_0211',
        'flyby': 'T83',
        'description': 'T83 flyby — Selk Crater region',
    },
]

bidr_dir = RAW_DIR / 'bidr'
bidr_dir.mkdir(exist_ok=True)

print("BIDR swath targets:")
for t in BIDR_TARGETS:
    print(f"  {t['volume']} ({t['flyby']}): {t['description']}")

print("\nNote: BIDR volumes are large. The download cells below will attempt")
print("to fetch only the BIDR S01 (primary beam) .IMG files.")
print("If automatic download fails, use the PDS Imaging Atlas:")
print("https://pds-imaging.jpl.nasa.gov/volumes/radar.html")

BIDR swath targets:
  CORADR_0048 (T8): T8 flyby — Shangri-La dune field coverage
  CORADR_0211 (T83): T83 flyby — Selk Crater region

Note: BIDR volumes are large. The download cells below will attempt
to fetch only the BIDR S01 (primary beam) .IMG files.
If automatic download fails, use the PDS Imaging Atlas:
https://pds-imaging.jpl.nasa.gov/volumes/radar.html


In [11]:
# Attempt to list and download BIDR products from PDS
# PDS directory structure: CORADR_XXXX/DATA/BIDR/

for target in BIDR_TARGETS:
    vol = target['volume']
    vol_dir = bidr_dir / vol
    vol_dir.mkdir(exist_ok=True)
    
    # Try to fetch the BIDR directory listing
    bidr_url = f"{PDS_BASE_URL}{vol}/DATA/BIDR/"
    print(f"\nChecking {bidr_url} ...")
    
    try:
        resp = requests.get(bidr_url, timeout=30)
        resp.raise_for_status()
        
        # Parse the directory listing for S01 BIDR .IMG files
        import re
        img_files = re.findall(r'href="(BIBQ.*S01[^"]*\.IMG)"', resp.text, re.IGNORECASE)
        if not img_files:
            # Try broader pattern
            img_files = re.findall(r'href="(BI[^"]*\.IMG)"', resp.text, re.IGNORECASE)
        
        print(f"  Found {len(img_files)} .IMG files")
        for fname in img_files[:5]:  # Show first 5
            print(f"    {fname}")
        if len(img_files) > 5:
            print(f"    ... and {len(img_files) - 5} more")
        
        # Download the first S01 primary beam file (usually the main SAR image)
        s01_files = [f for f in img_files if 'S01' in f.upper()]
        if s01_files:
            for fname in s01_files[:1]:  # Just the first one to start
                file_url = f"{bidr_url}{fname}"
                dest = vol_dir / fname
                try:
                    download_file(file_url, dest, description=f"{vol} {fname}")
                except Exception as e:
                    log.warning(f"Failed to download {fname}: {e}")
        else:
            log.info(f"No S01 files found for {vol}. Available: {img_files[:3]}")
            
    except Exception as e:
        log.warning(f"Could not access {bidr_url}: {e}")
        log.info(f"Manual download required from PDS for {vol}")


Checking https://pds-imaging.jpl.nasa.gov/data/cassini/cassini_orbiter/CORADR_0048/DATA/BIDR/ ...


08:41:53 [01_data_acquisition] WARNING: Could not access https://pds-imaging.jpl.nasa.gov/data/cassini/cassini_orbiter/CORADR_0048/DATA/BIDR/: 404 Client Error: Not Found for url: https://planetarydata.jpl.nasa.gov/img/data/cassini/cassini_orbiter/CORADR_0048/DATA/BIDR/


08:41:53 [01_data_acquisition] INFO: Manual download required from PDS for CORADR_0048



Checking https://pds-imaging.jpl.nasa.gov/data/cassini/cassini_orbiter/CORADR_0211/DATA/BIDR/ ...


08:41:55 [01_data_acquisition] INFO: No S01 files found for CORADR_0211. Available: []


  Found 0 .IMG files


## 5. Build Provenance Manifest

In [12]:
# Build MANIFEST.json logging all downloaded files
manifest = {
    'created': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'description': 'Titan SAR terrain classification — raw data provenance',
    'files': []
}

# Walk data/raw/ and log every file
for fpath in sorted(RAW_DIR.rglob('*')):
    if fpath.is_file() and fpath.name != 'MANIFEST.json':
        rel_path = str(fpath.relative_to(RAW_DIR))
        size = fpath.stat().st_size
        
        # Determine source URL
        source_url = 'unknown'
        description = ''
        if 'SAR_HiSAR' in fpath.name:
            source_url = SAR_MOSAIC_URL
            description = 'USGS SAR-HiSAR Global Mosaic, 351 m/pixel'
        elif 'nldsar' in str(rel_path).lower():
            source_url = f'https://zenodo.org/records/{NLDSAR_ZENODO_RECORD}'
            description = 'NLDSAR denoised SAR data (Lucas et al. 2014)'
        elif 'geomorph' in str(rel_path).lower() or 'geol' in str(rel_path).lower():
            source_url = 'https://astrogeology.usgs.gov'
            description = 'Geomorphological/geological map data'
        elif 'bidr' in str(rel_path).lower() or 'CORADR' in str(rel_path):
            source_url = PDS_BASE_URL
            description = 'Cassini BIDR SAR swath'
        
        # Compute checksum (skip for very large files > 2GB)
        if size < 2e9:
            checksum = sha256_file(fpath)
        else:
            checksum = 'skipped-large-file'
        
        manifest['files'].append({
            'path': rel_path,
            'size_bytes': size,
            'sha256': checksum,
            'source_url': source_url,
            'description': description,
            'access_date': datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%d'),
        })

save_manifest(manifest)
print(f"Manifest saved with {len(manifest['files'])} files.")
for f in manifest['files']:
    print(f"  {f['path']:50s}  {f['size_bytes']:>12,} bytes")

Manifest saved with 121 files.
  Titan_SAR_HiSAR_Global_Mosaic_351m.tif              1,061,868,159 bytes
  geomorphology/TITAN_2019-11_global_geomap_6unit.zip  2,363,135,078 bytes
  geomorphology/titan_6unit_geomap/TITAN_2019-11_global_geomap_6unit/Baseimages/Titan_ISS_Globe_65Sto45N_450M_AvgMos.tif  1,580,328,895 bytes
  geomorphology/titan_6unit_geomap/TITAN_2019-11_global_geomap_6unit/Baseimages/Titan_ISS_Globe_65Sto45N_450M_AvgMos.tif.aux.xml         2,259 bytes
  geomorphology/titan_6unit_geomap/TITAN_2019-11_global_geomap_6unit/Baseimages/Titan_ISS_Globe_65Sto45N_450M_AvgMos.tif.ovr   389,500,077 bytes
  geomorphology/titan_6unit_geomap/TITAN_2019-11_global_geomap_6unit/Baseimages/Titan_ISS_Globe_65Sto45N_450M_AvgMos.tif.xml           566 bytes
  geomorphology/titan_6unit_geomap/TITAN_2019-11_global_geomap_6unit/Baseimages/Titan_ISS_P19658_Mosaic_Global_4km.tif     8,178,362 bytes
  geomorphology/titan_6unit_geomap/TITAN_2019-11_global_geomap_6unit/Baseimages/Titan_ISS_P19658_Mos

## 6. Verification Summary

In [13]:
print("=" * 60)
print("DATA ACQUISITION SUMMARY")
print("=" * 60)

checks = {
    'SAR Mosaic': sar_mosaic_path.exists(),
    'NLDSAR Data': any(nldsar_dir.iterdir()) if nldsar_dir.exists() else False,
    'Geomorphological Map (GDB)': gdb_path.exists() if 'gdb_path' in dir() else False,
    'BIDR Swaths': any(bidr_dir.rglob('*.IMG')) if bidr_dir.exists() else False,
}

for item, ok in checks.items():
    status = 'OK' if ok else 'MISSING'
    print(f"  [{status:7s}] {item}")

if all(checks.values()):
    print("\nAll data acquired. Ready for Notebook 02.")
else:
    print("\nSome sources may need manual intervention — see notes above.")

DATA ACQUISITION SUMMARY
  [OK     ] SAR Mosaic
  [OK     ] NLDSAR Data
  [OK     ] Geomorphological Map (GDB)
  [MISSING] BIDR Swaths

Some sources may need manual intervention — see notes above.
